# Build and compare from a labeled export



This notebook takes one ZIP exported by **Analyzer Workshop**, builds a zero-shot forms analyzer and a learned analyzer with the same GPT-5.2 schema, then scores both against reviewed evaluation values.



It uses Content Understanding `2026-06-01-preview`. Training files are copied to a temporary, host-owned Blob container because the Content Understanding labeled-data contract requires Blob storage. Preview training distills the labeled patterns into the analyzer, so the temporary container is deleted immediately after the build completes.



Prerequisites: a Content Understanding endpoint, `DefaultAzureCredential` access, and Storage Blob Data Contributor on a host storage account. The person who labeled the documents does not need Content Understanding Studio, a Studio project, storage credentials, or a storage account.

In [ ]:
# Install once per environment; rerunning is safe.

%pip install -q "azure-ai-contentunderstanding>=1.1.0" "azure-identity>=1.17.0" "azure-storage-blob>=12.20.0" "pandas>=2.2.0" "requests>=2.32.0" "python-dotenv>=1.0.0"



from copy import deepcopy

from datetime import datetime, timedelta, timezone

import json

import os

from pathlib import Path, PurePosixPath

import re

import uuid

import zipfile



import pandas as pd

import requests

from azure.ai.contentunderstanding import ContentUnderstandingClient

from azure.core.exceptions import HttpResponseError, ResourceNotFoundError

from azure.identity import DefaultAzureCredential

from azure.storage.blob import (

    BlobServiceClient,

    ContainerSasPermissions,

    ContentSettings,

    generate_container_sas,

)

from dotenv import load_dotenv



search_roots = [Path.cwd(), *Path.cwd().parents]

REPO_ROOT = next(

    (root for root in search_roots if (root / "apps" / "workshop" / "api").is_dir()),

    Path.cwd(),

)

for env_path in (

    REPO_ROOT / ".env",

    REPO_ROOT / "apps" / "workshop" / "api" / ".env",

    REPO_ROOT / "demo" / "sov" / ".env",

    REPO_ROOT / "demo" / "labeled-analyzer" / ".env",

):

    if env_path.is_file():

        load_dotenv(env_path, override=False)



API_VERSION = "2026-06-01-preview"

LABELED_EXPORT = Path(r"C:\path\to\exported-labeled-data.zip")

TEST_DOCUMENT_NAME = ""  # Blank selects the first reviewed evaluation PDF.

STORAGE_ACCOUNT = os.getenv("LABELED_ANALYZER_STORAGE_ACCOUNT") or os.getenv("STORAGE_ACCOUNT_NAME")

ENDPOINT = (

    os.getenv("APP_CONTENT_UNDERSTANDING_ENDPOINT")

    or os.getenv("CONTENTUNDERSTANDING_ENDPOINT")

    or os.getenv("FOUNDRY_ENDPOINT")

)

GPT52_DEPLOYMENT = os.getenv("GPT52_MODEL_DEPLOYMENT", "gpt-5.2")

EMBEDDING_DEPLOYMENT = os.getenv("EMBEDDING_MODEL_DEPLOYMENT", "text-embedding-3-large")

CONFIGURE_DEFAULTS = True

RUN_AZURE = False



print(f"API: {API_VERSION}")

print(f"Export: {LABELED_EXPORT}")

print(f"Model deployments: gpt-5.2 -> {GPT52_DEPLOYMENT}; text-embedding-3-large -> {EMBEDDING_DEPLOYMENT}")

In [ ]:
assert LABELED_EXPORT.is_file(), f"Set LABELED_EXPORT to an Analyzer Workshop ZIP: {LABELED_EXPORT}"



with zipfile.ZipFile(LABELED_EXPORT) as archive:

    members = set(archive.namelist())

    manifest = json.loads(archive.read("manifest.json"))

    analyzer_template = json.loads(archive.read("analyzer-template.json"))



    def triplets(role):

        pdfs = sorted(

            name for name in members

            if name.startswith(f"{role}/") and name.lower().endswith(".pdf")

        )

        for pdf in pdfs:

            assert f"{pdf}.labels.json" in members, f"Missing labels for {pdf}"

            assert f"{pdf}.result.json" in members, f"Missing OCR result for {pdf}"

        return pdfs



    training_pdfs = triplets("training")

    evaluation_pdfs = triplets("evaluation")

    assert training_pdfs, "The export has no reviewed training documents."

    assert evaluation_pdfs, "The export has no reviewed evaluation documents."



    if TEST_DOCUMENT_NAME:

        test_member = f"evaluation/{PurePosixPath(TEST_DOCUMENT_NAME).name}"

        assert test_member in evaluation_pdfs, f"Evaluation document not found: {TEST_DOCUMENT_NAME}"

    else:

        test_member = evaluation_pdfs[0]



    test_labels = json.loads(archive.read(f"{test_member}.labels.json"))["fieldLabels"]

    typed_keys = {

        "string": "valueString", "date": "valueDate", "time": "valueTime",

        "number": "valueNumber", "integer": "valueInteger", "boolean": "valueBoolean",

    }

    expected_values = {

        name: field.get(typed_keys[field["type"]]) for name, field in test_labels.items()

    }



dataset_summary = pd.DataFrame([

    {"Split": "Training", "Reviewed documents": len(training_pdfs)},

    {"Split": "Evaluation", "Reviewed documents": len(evaluation_pdfs)},

])

display(dataset_summary)


In [ ]:
from decimal import Decimal, InvalidOperation



VALUE_KEYS = {

    "string": ("valueString", "value_string"),

    "date": ("valueDate", "value_date"),

    "time": ("valueTime", "value_time"),

    "number": ("valueNumber", "value_number"),

    "integer": ("valueInteger", "value_integer"),

    "boolean": ("valueBoolean", "value_boolean"),

}



def as_dict(value):

    if hasattr(value, "as_dict"):

        return value.as_dict()

    if isinstance(value, dict):

        return value

    raise TypeError(f"Expected an SDK model or dictionary, got {type(value).__name__}")



def get_value(field):

    field = as_dict(field) if field else {}

    for key in VALUE_KEYS.get(str(field.get("type", "")).casefold(), ()):

        if key in field:

            return field[key]

    return field.get("value")



def analysis_body(payload):

    body = as_dict(payload)

    for key in ("result", "analyzeResult", "analyze_result"):

        nested = body.get(key)

        if isinstance(nested, dict):

            body = nested

    return body



def result_fields(payload, label):

    body = analysis_body(payload)

    contents = body.get("contents") or []

    fields = {}

    for content in contents:

        fields.update((as_dict(content).get("fields") or {}))

    if not fields:

        summary = {

            "top_level_keys": sorted(as_dict(payload)),

            "result_keys": sorted(body),

            "content_count": len(contents),

            "warnings": body.get("warnings"),

        }

        raise RuntimeError(f"{label} returned no fields. Response summary: {summary}")

    return fields



def configure_model_defaults(credential):

    token = credential.get_token("https://cognitiveservices.azure.com/.default").token

    url = f"{ENDPOINT.rstrip('/')}/contentunderstanding/defaults?api-version={API_VERSION}"

    headers = {"Authorization": f"Bearer {token}"}

    current_response = requests.get(url, headers=headers, timeout=30)

    current = current_response.json() if current_response.ok and current_response.text else {}

    deployments = dict(current.get("modelDeployments") or {})

    deployments.update({

        "gpt-5.2": GPT52_DEPLOYMENT,

        "text-embedding-3-large": EMBEDDING_DEPLOYMENT,

    })

    response = requests.patch(

        url,

        headers={**headers, "Content-Type": "application/merge-patch+json"},

        json={"modelDeployments": deployments},

        timeout=30,

    )

    response.raise_for_status()

    return deployments



def create_or_replace(client, analyzer_id, resource):

    allow_replace = False

    try:

        existing = as_dict(client.get_analyzer(analyzer_id=analyzer_id))

    except ResourceNotFoundError:

        existing = None

    if existing:

        expected_tags = resource.get("tags") or {}

        actual_tags = existing.get("tags") or {}

        if actual_tags.get("project") != expected_tags.get("project"):

            raise RuntimeError(f"Analyzer {analyzer_id!r} exists but is not owned by this export.")

        allow_replace = True

    client.begin_create_analyzer(

        analyzer_id=analyzer_id,

        resource=resource,

        allow_replace=allow_replace,

    ).result()

    return as_dict(client.get_analyzer(analyzer_id=analyzer_id))



def analyze(client, analyzer_id, content):

    result = client.begin_analyze_binary(

        analyzer_id=analyzer_id,

        binary_input=content,

        content_type="application/pdf",

    ).result()

    payload = as_dict(result)

    result_fields(payload, analyzer_id)

    return payload



def normalize(value):

    if value is None:

        return None

    if isinstance(value, (int, float, Decimal)):

        try:

            return Decimal(str(value))

        except InvalidOperation:

            pass

    return re.sub(r"\s+", " ", str(value)).strip().casefold()



def compare(expected, zero_payload, learned_payload):

    zero_fields = result_fields(zero_payload, "Zero-shot analyzer")

    learned_fields = result_fields(learned_payload, "Learned analyzer")

    rows = []

    for name, expected_value in expected.items():

        zero_field = as_dict(zero_fields.get(name)) if zero_fields.get(name) else {}

        learned_field = as_dict(learned_fields.get(name)) if learned_fields.get(name) else {}

        zero_value = get_value(zero_field)

        learned_value = get_value(learned_field)

        rows.append({

            "Field": name,

            "Reviewed value": expected_value,

            "Zero-shot": zero_value,

            "Zero confidence": zero_field.get("confidence"),

            "Zero correct": normalize(zero_value) == normalize(expected_value),

            "Learned": learned_value,

            "Learned confidence": learned_field.get("confidence"),

            "Learned correct": normalize(learned_value) == normalize(expected_value),

        })

    return pd.DataFrame(rows)


In [ ]:
assert RUN_AZURE, "Review the settings above, then set RUN_AZURE = True."

assert ENDPOINT, "Set APP_CONTENT_UNDERSTANDING_ENDPOINT."

assert STORAGE_ACCOUNT, "Set LABELED_ANALYZER_STORAGE_ACCOUNT or STORAGE_ACCOUNT_NAME."



credential = DefaultAzureCredential()

if CONFIGURE_DEFAULTS:

    configured = configure_model_defaults(credential)

    print("Content Understanding model defaults are ready:", {

        "gpt-5.2": configured["gpt-5.2"],

        "text-embedding-3-large": configured["text-embedding-3-large"],

    })



client = ContentUnderstandingClient(

    endpoint=ENDPOINT,

    credential=credential,

    api_version=API_VERSION,

)

blob_service = BlobServiceClient(

    account_url=f"https://{STORAGE_ACCOUNT}.blob.core.windows.net",

    credential=credential,

)

container_name = f"cu-nb-{uuid.uuid4().hex[:20]}"

container = blob_service.get_container_client(container_name)

container_created = False



base_analyzer_id = manifest["base_analyzer_id"]

learned_analyzer_id = manifest["trained_analyzer_id"]



try:

    try:

        container.create_container(metadata={

            "purpose": "temporary-labeled-training",

            "expires": str(int((datetime.now(timezone.utc) + timedelta(hours=3)).timestamp())),

        })

    except HttpResponseError as exc:

        if exc.status_code == 403:

            raise RuntimeError(

                "The notebook identity cannot create temporary training storage. "

                "Grant the host identity Storage Blob Data Contributor on the configured account, "

                "or use Analyzer Workshop on a host where that role is already configured."

            ) from exc

        raise

    container_created = True

    with zipfile.ZipFile(LABELED_EXPORT) as archive:

        for pdf in training_pdfs:

            for member in (pdf, f"{pdf}.labels.json", f"{pdf}.result.json"):

                content_type = "application/pdf" if member.endswith(".pdf") else "application/json"

                container.upload_blob(

                    member,

                    archive.read(member),

                    overwrite=True,

                    content_settings=ContentSettings(content_type=content_type),

                )



    now = datetime.now(timezone.utc)

    expires = now + timedelta(minutes=120)

    delegation_key = blob_service.get_user_delegation_key(now - timedelta(minutes=5), expires)

    sas = generate_container_sas(

        account_name=STORAGE_ACCOUNT,

        container_name=container_name,

        user_delegation_key=delegation_key,

        permission=ContainerSasPermissions(read=True, list=True),

        start=now - timedelta(minutes=5),

        expiry=expires,

    )

    container_url = f"https://{STORAGE_ACCOUNT}.blob.core.windows.net/{container_name}?{sas}"



    base_resource = deepcopy(analyzer_template)

    learned_resource = deepcopy(analyzer_template)

    learned_resource["knowledgeSources"] = [{

        "kind": "labeledData",

        "containerUrl": container_url,

        "prefix": "training/",

    }]



    base_metadata = create_or_replace(client, base_analyzer_id, base_resource)

    learned_metadata = create_or_replace(client, learned_analyzer_id, learned_resource)

finally:

    if container_created:

        container.delete_container()



print({

    "zero_shot": base_metadata.get("analyzerId", base_analyzer_id),

    "learned": learned_metadata.get("analyzerId", learned_analyzer_id),

    "temporary_staging": "deleted",

})


In [ ]:
with zipfile.ZipFile(LABELED_EXPORT) as archive:

    test_bytes = archive.read(test_member)



zero_response = analyze(client, base_analyzer_id, test_bytes)

learned_response = analyze(client, learned_analyzer_id, test_bytes)

comparison = compare(expected_values, zero_response, learned_response)



zero_accuracy = float(comparison["Zero correct"].mean())

learned_accuracy = float(comparison["Learned correct"].mean())

summary = pd.DataFrame([

    {"Analyzer": "Zero-shot", "Accuracy": zero_accuracy},

    {"Analyzer": "Learned", "Accuracy": learned_accuracy},

    {"Analyzer": "Delta", "Accuracy": learned_accuracy - zero_accuracy},

])



print(

    f"Zero-shot: {zero_accuracy:.1%} | Learned: {learned_accuracy:.1%} | "

    f"Delta: {(learned_accuracy - zero_accuracy) * 100:+.1f} pp"

)

display(comparison)

display(summary)



output_dir = LABELED_EXPORT.parent / "comparison-output"

output_dir.mkdir(exist_ok=True)

comparison.to_csv(output_dir / "field-comparison.csv", index=False)

(output_dir / "summary.json").write_text(

    json.dumps({

        "test_document": PurePosixPath(test_member).name,

        "zero_shot_accuracy": zero_accuracy,

        "learned_accuracy": learned_accuracy,

        "accuracy_delta_points": (learned_accuracy - zero_accuracy) * 100,

    }, indent=2),

    encoding="utf-8",

)

print(f"Saved comparison to {output_dir}")


## Interpret the result



Use accuracy against independently reviewed values as the decision criterion. A higher confidence score is not evidence of a better extraction when the value is wrong.



Labeled examples teach layout, terminology, and field-selection patterns. They do not repair OCR glyph mistakes. The `2026-06-01-preview` training path distills the labeled patterns into the built analyzer, which is why this notebook removes its temporary Blob staging container immediately after creation.